# Importing the libraries and the dataset

In [1]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor

In [2]:
df=pd.read_csv('truedata.csv')

In [3]:
df

,Strain Rate,Temperature,True Strain,True Stress,True Plastic Strain
0,0.0001,27.0,0.10683,939.31764,0.00030
1,0.0001,27.0,0.10683,939.32333,0.00030
2,0.0001,27.0,0.10684,939.33680,0.00031
3,0.0001,27.0,0.10684,939.34085,0.00031
4,0.0001,27.0,0.10685,939.34472,0.00032
...,...,...,...,...,...
514332,NaN,NaN,NaN,NaN,NaN
514333,NaN,NaN,NaN,NaN,NaN
514334,NaN,NaN,NaN,NaN,NaN
514335,NaN,NaN,NaN,NaN,NaN


In [4]:
df = df.dropna()

In [5]:
df

,Strain Rate,Temperature,True Strain,True Stress,True Plastic Strain
0,0.0001,27.0,0.10683,939.31764,0.00030
1,0.0001,27.0,0.10683,939.32333,0.00030
2,0.0001,27.0,0.10684,939.33680,0.00031
3,0.0001,27.0,0.10684,939.34085,0.00031
4,0.0001,27.0,0.10685,939.34472,0.00032
...,...,...,...,...,...
162754,0.0100,500.0,0.06196,597.07372,0.01804
162755,0.0100,500.0,0.06218,597.09635,0.01826
162756,0.0100,500.0,0.06239,597.11009,0.01847
162757,0.0100,500.0,0.06260,597.11561,0.01868


In [6]:
TargetVariable=['True Stress']
Predictors=['Strain Rate','Temperature','True Plastic Strain']
 
X=df[Predictors].values
y=df[TargetVariable].values

# Hyperparameter tuning with GridSearchCV where k=10

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test  = train_test_split(X, y, test_size=0.3, random_state=42)

In [8]:
max_depth=[3,6,9,12,15]
n_estimators = [10, 50, 100, 200, 500]
param_grid = dict(max_depth=max_depth, n_estimators=n_estimators)

# Build the gridsearch
dfrst = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth)
grid = GridSearchCV(estimator=dfrst, param_grid=param_grid, cv = 10,n_jobs=-1)
grid_results = grid.fit(X_train, y_train)

C:\Users\ankit\Downloads\Anaconda\lib\site-packages\sklearn\model_selection\_search.py:926: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  self.best_estimator_.fit(X, y, **fit_params)


In [9]:
print("Best parameters:", grid_results.best_params_)

Best parameters: {'max_depth': 15, 'n_estimators': 100}


In [10]:
grid_results.best_estimator_

RandomForestRegressor(max_depth=15)

# Refitting and retraining the model using the best parameters and evaluating its performance

In [9]:
rf = RandomForestRegressor(max_depth=15, n_estimators=100)
rf.fit(X_train,y_train)

C:\Users\ankit\AppData\Local\Temp\ipykernel_35308\1601727070.py:2: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  rf.fit(X_train,y_train)


RandomForestRegressor(max_depth=15)

In [10]:
y_pred = rf.predict(X_test)

In [11]:
y_pred

array([ 781.83212465,  622.22043036,  691.64220997, ...,  745.43945853,
       1152.28417699,  950.01703668])

In [12]:
y1_test = y_test.ravel()

In [13]:
y1_test

array([ 781.83656,  622.26907,  694.36165, ...,  740.82561, 1152.24326,
        950.02404])

In [19]:
y_pred

array([ 781.83212465,  622.22043036,  691.64220997, ...,  745.43945853,
       1152.28417699,  950.01703668])

In [14]:
APE=100*(abs(y1_test-y_pred)/y1_test)

In [15]:
print('The Accuracy of RFR model is:', np.mean(APE))

The Accuracy of RFR model is: 0.11932555865195453


In [16]:
MAE=(abs(y1_test-y_pred))
print('The Accuracy of RFR model is:',np.mean(MAE))

The Accuracy of RFR model is: 0.8506601081106783


In [18]:
import statistics
var = (statistics.variance(y_pred))
print(var)
chi_sq = np.sum(((y1_test-y_pred)**2)/var)
red_chi_sq = chi_sq/5074
print('The reduced chi squared value for RFR is', red_chi_sq)

23043.6890892721
The reduced chi squared value for RFR is 0.0020966403447208516


# Validating the model

In [22]:
dfvalid=pd.read_csv('150Cvalidationdata.csv')

In [23]:
Predictors=['Strain Rate','Temperature','True Plastic Strain']
X_valid=dfvalid[Predictors].values

In [24]:
X_valid

array([[1.0000000e-03, 1.5000000e+02, 6.1896600e-05],
       [1.0000000e-03, 1.5000000e+02, 5.2687700e-05],
       [1.0000000e-03, 1.5000000e+02, 6.9798000e-05],
       ...,
       [1.0000000e-03, 1.5000000e+02, 9.4723457e-02],
       [1.0000000e-03, 1.5000000e+02, 9.4695481e-02],
       [1.0000000e-03, 1.5000000e+02, 9.4743924e-02]])

In [25]:
y_pred_valid = rf.predict(X_valid)

In [26]:
y_pred_valid

array([777.75436127, 777.75436127, 777.75436127, ..., 837.73286707,
       837.70603168, 837.73402251])

In [27]:
import pandas as pd 
pd.DataFrame(y_pred_valid).to_csv("rfrvalidationpredicteddata.csv")